# 9. 분류에서 객체 탐지로

이 노트북은 `08_데이터_증강과_일반화.ipynb` 다음 단계로, **이미지 분류(classification)에서 객체 탐지(object detection)로 넘어가기 위한 개념적 다리** 를 만드는 것이 목표입니다.

1장에서는 주로 `이 이미지가 무엇인가?` 라는 질문을 다뤘습니다. 이제 2장에서는 질문이 바뀝니다.

- 분류: `이 이미지의 대표 클래스는 무엇인가?`
- 탐지: `무엇이 어디에 있는가?`

즉, 탐지는 **클래스 분류 + 위치 추정(localization)** 이 동시에 필요한 문제입니다.

이번 노트북의 목표는 다음과 같습니다.

- 이미지 분류와 객체 탐지의 차이를 명확히 이해합니다.
- 탐지가 왜 더 어려운 문제인지 직관적으로 정리합니다.
- 탐지 라벨이 왜 bounding box를 필요로 하는지 이해합니다.
- 다음 노트북의 `Bounding Box`, `IoU`, `NMS` 개념으로 자연스럽게 넘어갈 준비를 합니다.


## 9-1. 분류 문제는 무엇을 출력할까?

이미지 분류 모델은 보통 **이미지 전체를 하나의 입력** 으로 보고, 그 이미지가 어떤 클래스에 속하는지를 하나의 벡터로 출력합니다.

예를 들어 고양이 사진 한 장이 들어오면, 모델은 다음과 같이 생각합니다.

- 입력: 이미지 1장
- 출력: `cat`, `dog`, `car` 같은 클래스 확률

이때 중요한 점은, 분류 모델은 보통 **객체가 이미지의 어디에 있는지 명시적으로 말하지 않아도 된다** 는 것입니다. 이미지 안에 고양이가 왼쪽 아래에 있든, 중앙에 있든, 조금 작게 있든, 최종 목적은 `고양이`라고 맞히는 것입니다.


In [ ]:
from pprint import pprint

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
classification_label = {
    'image_id': 'sample_001',
    'label': 'cat'
}

detection_label = {
    'image_id': 'sample_001',
    'objects': [
        {'class': 'cat', 'bbox_xywh': [40, 30, 80, 70]},
        {'class': 'dog', 'bbox_xywh': [135, 80, 70, 90]}
    ]
}

print('[분류 라벨 예시]')
pprint(classification_label)

print('\n[탐지 라벨 예시]')
pprint(detection_label)


위 출력만 봐도 차이가 선명합니다.

- 분류 라벨은 보통 이미지당 클래스 하나면 충분합니다.
- 탐지 라벨은 이미지 안의 **각 객체마다 클래스와 위치 정보** 가 필요합니다.

이 차이 하나 때문에 데이터셋 구성, 모델 출력, 손실 함수, 평가 방식이 전부 달라집니다.


## 9-2. 같은 이미지도 분류와 탐지에서는 목표가 다르다

아래 예시는 하나의 장면 안에 두 개의 객체가 들어 있는 상황을 단순화해서 그린 것입니다.

- 분류 관점: `이 장면의 대표 클래스는 무엇인가?`
- 탐지 관점: `고양이와 개가 각각 어디에 있는가?`

즉, 탐지는 이미지 전체를 한 번에 요약하는 것이 아니라, **장면 안의 여러 객체를 개별적으로 찾아야 합니다.**


In [ ]:
def draw_scene(ax, boxes=None, title=''):
    ax.set_xlim(0, 240)
    ax.set_ylim(180, 0)
    ax.set_facecolor('#f7f7f7')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

    # 간단한 배경 요소
    ax.add_patch(Rectangle((0, 120), 240, 60, color='#ddebd3', alpha=0.9))
    ax.add_patch(Rectangle((0, 0), 240, 120, color='#eaf4ff', alpha=1.0))

    cat_box = (35, 40, 75, 65)
    dog_box = (135, 70, 70, 85)

    ax.add_patch(Rectangle((cat_box[0], cat_box[1]), cat_box[2], cat_box[3], color='#f2b880', alpha=0.95))
    ax.text(cat_box[0] + 8, cat_box[1] + 35, 'cat', fontsize=12, weight='bold')

    ax.add_patch(Rectangle((dog_box[0], dog_box[1]), dog_box[2], dog_box[3], color='#8fb7ff', alpha=0.95))
    ax.text(dog_box[0] + 10, dog_box[1] + 45, 'dog', fontsize=12, weight='bold')

    if boxes:
        for x, y, w, h, label, color in boxes:
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
            ax.text(x, max(12, y - 5), label, color=color, fontsize=11, weight='bold')


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
draw_scene(axes[0], title='분류 관점: 이 장면은 무엇인가?')
axes[0].text(8, 18, '예시 출력: pet scene / cat', fontsize=12, weight='bold', color='black')

draw_scene(
    axes[1],
    boxes=[
        (35, 40, 75, 65, 'cat', 'crimson'),
        (135, 70, 70, 85, 'dog', 'royalblue')
    ],
    title='탐지 관점: 무엇이 어디에 있는가?'
)

plt.tight_layout()
plt.show()


왼쪽처럼 분류만 한다면, 모델은 장면 전체를 보고 하나의 답을 내놓아도 됩니다. 하지만 오른쪽처럼 탐지를 하려면 적어도 다음 두 가지가 필요합니다.

- 객체의 종류가 무엇인지
- 그 객체가 이미지의 어느 위치에 있는지

그래서 객체 탐지는 흔히 **classification + localization** 문제라고 설명합니다.


## 9-3. 왜 객체 탐지는 더 어려울까?

탐지가 분류보다 어려운 이유는 단순히 출력이 많아서만은 아닙니다. 실제로는 아래 문제가 함께 들어 있습니다.

1. **객체 수가 고정되지 않습니다.**
   이미지마다 객체가 0개일 수도, 1개일 수도, 여러 개일 수도 있습니다.
2. **위치를 맞혀야 합니다.**
   클래스만 맞는다고 끝이 아니라, 박스가 실제 물체 위치와 충분히 겹쳐야 합니다.
3. **크기와 비율이 다양합니다.**
   작은 객체, 큰 객체, 길쭉한 객체가 섞여 있을 수 있습니다.
4. **가려짐(occlusion)과 배경 복잡도가 큽니다.**
   일부만 보이는 객체나 복잡한 배경 속 객체는 더 어렵습니다.
5. **여러 객체가 서로 가까이 붙어 있을 수 있습니다.**
   같은 클래스가 여러 개 있을 때 중복 박스를 어떻게 정리할지도 중요합니다.


In [ ]:
def sliding_window_count(image_size, window_size, stride):
    width, height = image_size
    win_w, win_h = window_size
    count_x = ((width - win_w) // stride) + 1
    count_y = ((height - win_h) // stride) + 1
    return count_x * count_y


image_size = (256, 256)
window_sizes = [(32, 32), (64, 64), (128, 128)]
stride = 16

for ws in window_sizes:
    num_windows = sliding_window_count(image_size, ws, stride)
    print(f'window={ws}, stride={stride} -> 후보 개수: {num_windows}')


위 숫자는 아주 단순한 예시지만, 탐지가 왜 어려운지 감을 줍니다. 과거 방식 중 하나였던 `sliding window`는 여러 위치와 크기에서 계속 창(window)을 움직이며 객체 후보를 검사했습니다.

문제는 후보 수가 빠르게 커진다는 점입니다. 크기와 종횡비를 더 늘리고, 더 작은 stride를 쓰면 계산량은 훨씬 커집니다. YOLO 같은 현대 탐지 모델이 주목받은 이유도, 이런 비효율을 크게 줄였기 때문입니다.


## 9-4. 탐지에서는 정답도 박스로 표현된다

분류에서는 정답이 `cat`처럼 클래스 하나면 끝이지만, 탐지에서는 보통 다음 형태가 필요합니다.

- 클래스 라벨
- bounding box 좌표

가장 흔한 표현 중 하나는 `xywh` 형식입니다.

- `x, y`: 박스의 왼쪽 위 좌표
- `w, h`: 박스의 너비와 높이

또는 `x1, y1, x2, y2`처럼 양쪽 꼭짓점을 쓰기도 합니다. 다음 노트북에서는 이 박스 표현과 `IoU(Intersection over Union)`를 더 정확히 다룹니다.


In [ ]:
def xywh_to_xyxy(box):
    x, y, w, h = box
    return (x, y, x + w, y + h)


def compute_iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


gt_box_xywh = (40, 30, 80, 70)
pred_good_xywh = (45, 35, 78, 68)
pred_bad_xywh = (110, 40, 70, 60)

gt_xyxy = xywh_to_xyxy(gt_box_xywh)
good_xyxy = xywh_to_xyxy(pred_good_xywh)
bad_xyxy = xywh_to_xyxy(pred_bad_xywh)

print('정답 박스:', gt_xyxy)
print('잘 맞춘 예측 IoU:', round(compute_iou_xyxy(gt_xyxy, good_xyxy), 3))
print('엇나간 예측 IoU:', round(compute_iou_xyxy(gt_xyxy, bad_xyxy), 3))


여기서 중요한 직관은 이것입니다.

- 분류에서는 정답 클래스만 맞히면 됩니다.
- 탐지에서는 **클래스도 맞고, 위치도 충분히 맞아야 합니다.**

그래서 탐지 평가는 단순 정확도 하나로 끝나지 않고, `IoU`, `precision`, `recall`, `mAP` 같은 개념이 등장합니다.


## 9-5. 모델 출력도 달라진다

분류 모델과 탐지 모델의 출력 형식을 단순화해서 비교하면 다음과 같습니다.


In [ ]:
comparison_rows = [
    ('문제 정의', '이미지가 무엇인가?', '무엇이 어디에 있는가?'),
    ('출력 개수', '보통 이미지당 1개', '이미지당 여러 개 가능'),
    ('라벨 형식', '클래스 하나', '클래스 + 박스 좌표'),
    ('평가 핵심', '정확도', '정확도 + 위치 일치도'),
    ('대표 질문', 'cat 인가?', 'cat 이 어디 있는가?')
]

header = f"{'항목':<12} | {'분류':<22} | {'탐지':<26}"
print(header)
print('-' * len(header))
for row in comparison_rows:
    print(f'{row[0]:<12} | {row[1]:<22} | {row[2]:<26}')


실제로는 탐지 모델도 내부적으로는 분류 성분을 포함합니다. 다만 거기에 더해 **박스 회귀(box regression)** 나, 후보 박스 중 어떤 것을 남길지 정하는 후처리까지 필요합니다. 그래서 탐지는 분류보다 구조가 더 복합적입니다.


## 9-6. 정리

이번 노트북에서 정리한 핵심은 다음과 같습니다.

- 이미지 분류는 보통 `이미지 전체에 대한 하나의 클래스 판단` 문제입니다.
- 객체 탐지는 `클래스 판단 + 위치 추정` 이 동시에 필요한 문제입니다.
- 탐지는 여러 객체, 다양한 크기, 복잡한 배경, 위치 오차까지 함께 다뤄야 하므로 더 어렵습니다.
- 그래서 탐지 데이터셋은 클래스뿐 아니라 `bounding box` 라벨이 필요합니다.

다음 노트북 `10_Bounding_Box와_IoU.ipynb`에서는 바로 이 위치 표현을 더 엄밀하게 다룹니다. 특히 아래 개념이 이어집니다.

- bounding box 좌표 형식
- IoU가 무엇인지
- confidence score를 어떻게 이해해야 하는지

즉, 이번 노트북은 **"왜 탐지에서는 박스가 필요한가?"** 를 이해하는 브리지이고, 다음 노트북부터는 그 박스를 수학적으로 다루기 시작합니다.
